# Vulcan validation and troubleshooting — v19

Companion to `Large_FOV_Nuclear_Pipeline_v19.ipynb`. Nothing here changes the pipeline's outputs:
every cell reads the pipeline run and the hand-traced gold ROIs and writes only to
`<run>/qc/validation_v19/`.

The pipeline's definition cells (tag `v19-def`) and bootstrap cell (tag `v19-bootstrap`) are loaded
from the pipeline notebook file, so this notebook runs **the same code and the same configuration**
and finds the same run directory. Run the pipeline through §20 first.

| § | content | purpose |
|---|---|---|
| V0 | load definitions, run, tables | — |
| V1 | normalisation parity with the training patches | input contract check |
| V2 | gold ROIs → maximal sections → same rules as the pipeline; **cluster flags** written | gold reference |
| V3 | median logistic: model vs gold, equivalence on A(30)/A(55), phase velocities | **validation result** |
| V4 | matched decomposition of the population gap | where a gap comes from |
| V5 | equator classes per gold nucleus; unmatched model objects and gallery | detection audit |
| V6 | growth-model family comparison (AIC, leave-one-frame-out CV) | curve choice |
| V7 | selection-rule sensitivity | robustness of the rules |
| V8 | Vulcan 3.1.2 hard-example export | next training round |

Gold conventions: touching nuclei traced as one ROI are **flagged** (V2) and excluded from both
sides by the same single-nucleus rule as the model; the ROI archive itself is never modified.

In [ ]:
# V0. Load the pipeline's definitions and bootstrap from the pipeline notebook file.
import json, __future__
from pathlib import Path
PIPELINE_NOTEBOOK = (Path.home() / "Projects/Nuclear_Scaling/Code_Repository/Python/Image_Segmentation"
                     / "Large_FOV_Nuclear_Pipeline_v19.ipynb")
_nb = json.loads(PIPELINE_NOTEBOOK.read_text())
_loaded = 0
for _tag in ("v19-def", "v19-bootstrap"):
    for _c in _nb["cells"]:
        if _c["cell_type"] == "code" and _tag in _c.get("metadata", {}).get("tags", []):
            exec(compile("".join(_c["source"]), f"pipeline:{_tag}:{_c.get('id', '')}", "exec",
                         flags=__future__.annotations.compiler_flag, dont_inherit=True), globals())
            _loaded += 1
print(f"loaded {_loaded} cells from {PIPELINE_NOTEBOOK.name}")
VALID_DIR = cfg.qc_dir / "validation_v19"
VALID_DIR.mkdir(parents=True, exist_ok=True)
print("run:", cfg.output_dir, "\nvalidation outputs:", VALID_DIR)


In [ ]:
# V0b. Image, saved masks and the pipeline's tables for this run (read-only).
img_5d = load_image_5d(cfg.input_image_path)
T_, Z_, _, H, W = img_5d.shape
inst = tiff.memmap(str(cfg.nucleus_instance_hyperstack_path), mode="r")
heads_mm = tiff.memmap(str(full_head_path(cfg)), mode="r")              # (T, Z, 8, H, W)
sections_df = pd.read_pickle(cfg.obj_dir / "sections_all.pkl")
single_nuclei_df = pd.read_pickle(cfg.exports_dir / "single_nuclei" / "single_nuclei.pkl")
POPULATION = dpop.load_population(Path(sorted(
    (cfg.analysis_dir / "selected_population" / "populations").glob("pop-*"), key=lambda p: p.stat().st_mtime)[-1]))
print("image", img_5d.shape, "| sections", len(sections_df), "| selected nuclei", len(single_nuclei_df),
      "| population", POPULATION["tag"])


## V1. Normalisation parity

In [ ]:
# V1. ONE-OFF CHECK (informational, changes nothing): does the pipeline normalise exactly like the
# training patches? Compares normalization_stats_for_timepoint (inference) with the gold pool's
# qc/norm_stats.json (training). They should agree to rounding.
NORM_STATS_JSON = None   # None = search Training_Data (reviewed pool preferred)
if NORM_STATS_JSON is None:
    _cands = sorted(paths.training_data_dir.glob("*/qc/norm_stats.json"), key=lambda p: "review" not in str(p).lower())
    print("candidates:", *_cands, sep="\n  ")
    NORM_STATS_JSON = _cands[0] if _cands else None
if NORM_STATS_JSON is None:
    print("No norm_stats.json found; set NORM_STATS_JSON by hand.")
else:
    _train = {tuple(map(int, k.split("_"))): v for k, v in json.loads(Path(NORM_STATS_JSON).read_text()).items()}
    _rows = []
    for _t in range(T_):
        _mine = normalization_stats_for_timepoint(img_5d, _t, cfg)[:3]
        for _c, (_lo, _hi) in zip([cfg.nuclear_channel_index, cfg.npc_channel_index, cfg.membrane_channel_index], _mine):
            _tlo, _thi = _train.get((_t, _c), (np.nan, np.nan))
            _rows.append(dict(t=_t, channel=_c, pipe_lo=_lo, train_lo=_tlo, pipe_hi=_hi, train_hi=_thi,
                              rel_diff_hi=abs(_hi - _thi) / max(abs(_thi), 1e-9)))
    _norm = pd.DataFrame(_rows)
    print("using:", NORM_STATS_JSON, "| max relative difference in p99.8:", f"{_norm.rel_diff_hi.max():.2e}",
          "-> OK" if _norm.rel_diff_hi.max() < 1e-3 else "-> MISMATCH")
    display(_norm.round(2))


## V2. Gold reference

In [ ]:
# V2a. Gold ROIs: read and rasterise every traced outline (one per t, z), then link outlines of
# the same nucleus across planes using gold only (union-find; overlap >= 30 % of the smaller
# outline; up to one missing plane). Gold has no droplet motion problem: it is traced per plane.
import roifile, cv2
GOLD_TRAIN_RUN = Path("/home/tdeibert/Projects/Nuclear_Scaling/Models/runs/Vulcan_3.1.1_iezqw_2d87f0ec3c")
GOLD_LINK_FRAC, GOLD_LINK_GAP = 0.30, 1
GOLD_MIN_PLANES = 3      # a traced nucleus must span >= 3 eligible planes (gold-only requirement)

_rec = json.loads((GOLD_TRAIN_RUN / "phase3_run.json").read_text())
_pool = Path(_rec["pools"]["gold"]["root"])
ROI_PATH = Path(json.loads((_pool / "manifest.json").read_text())["roi_paths"]["nuc"])
print("gold ROIs:", ROI_PATH)

rois = []
for o in roifile.roiread(str(ROI_PATH)):
    t, z = int(o.t_position) - 1, int(o.z_position) - 1
    p = np.asarray(o.coordinates(), float)
    y0, x0 = int(np.floor(p[:, 1].min())), int(np.floor(p[:, 0].min()))
    y1, x1 = int(np.ceil(p[:, 1].max())) + 1, int(np.ceil(p[:, 0].max())) + 1
    m = np.zeros((y1 - y0, x1 - x0), np.uint8)
    cv2.fillPoly(m, [np.round(p - [x0, y0]).astype(np.int32)], 1)
    m = m.astype(bool); yy, xx = np.nonzero(m)
    rois.append(dict(t=t, z=z, box=(y0, x0, y1, x1), mask=m, area_px=int(m.sum()),
                     cy=y0 + yy.mean(), cx=x0 + xx.mean()))
G = pd.DataFrame(rois)
gz = G.groupby(["t", "z"]).groups
print(f"{len(G)} gold outlines, t {G.t.min()}–{G.t.max()}, z {G.z.min()}–{G.z.max()}")


def _overlap(a, b):
    (ay0, ax0, ay1, ax1), (by0, bx0, by1, bx1) = a["box"], b["box"]
    y0, x0, y1, x1 = max(ay0, by0), max(ax0, bx0), min(ay1, by1), min(ax1, bx1)
    if y0 >= y1 or x0 >= x1:
        return 0
    return int((a["mask"][y0-ay0:y1-ay0, x0-ax0:x1-ax0] & b["mask"][y0-by0:y1-by0, x0-bx0:x1-bx0]).sum())


_parent = list(range(len(G)))
def _find(i):
    while _parent[i] != i:
        _parent[i] = _parent[_parent[i]]; i = _parent[i]
    return i
for _t, g in G.groupby("t"):
    idx = g.index.to_list()
    for i in idx:
        for j in idx:
            if j <= i or not (1 <= abs(rois[i]["z"] - rois[j]["z"]) <= GOLD_LINK_GAP + 1):
                continue
            ov = _overlap(rois[i], rois[j])
            if ov and ov >= GOLD_LINK_FRAC * min(rois[i]["area_px"], rois[j]["area_px"]):
                _parent[_find(i)] = _find(j)
G["gold_id"] = [_find(i) for i in range(len(G))]


In [ ]:
# V2b. Gold maximal sections, and the SAME rules as the pipeline (§16): timepoints, eligible planes,
# edge, area floor, single-nucleus (solidity / another traced outline within sel_touch_px on the plane).
# Writes the gold cluster flag table; the ROI archive is never modified.
E = G[G.z >= cfg.focus_min_z]
span = E.groupby("gold_id").z.nunique().rename("n_planes")
gmax = E.sort_values("area_px", ascending=False).drop_duplicates("gold_id").join(span, on="gold_id")
gmax = gmax[gmax.t >= cfg.sel_min_timepoint].copy()
gmax["roi_index"] = gmax.index
gmax["area_um2"] = gmax.area_px * cfg.pixel_size_um ** 2
gmax["centroid_x_px"], gmax["centroid_y_px"] = gmax.cx, gmax.cy
b = np.array(list(gmax.box))
gmax["bbox_min_row"], gmax["bbox_min_col"], gmax["bbox_max_row"], gmax["bbox_max_col"] = b.T
gmax = add_tile_timing_metadata(gmax, (H, W), cfg)

disk = morphology.disk(cfg.sel_touch_px)
sol, touch = [], []
for _, g in gmax.iterrows():
    roi = rois[g.roi_index]; (a, b_, c, d), m = roi["box"], roi["mask"]
    sol.append(measure.regionprops(m.astype(np.uint8))[0].solidity)
    grown = ndi.binary_dilation(np.pad(m, cfg.sel_touch_px), structure=disk)
    A, B = a - cfg.sel_touch_px, b_ - cfg.sel_touch_px
    hit = False
    for k in gz.get((g.t, g.z), []):
        if k == g.roi_index:
            continue
        (oa, ob, oc, od), om = rois[k]["box"], rois[k]["mask"]
        y0, x0 = max(A, oa), max(B, ob)
        y1, x1 = min(A + grown.shape[0], oc), min(B + grown.shape[1], od)
        if y0 < y1 and x0 < x1 and (grown[y0 - A:y1 - A, x0 - B:x1 - B] & om[y0 - oa:y1 - oa, x0 - ob:x1 - ob]).any():
            hit = True; break
    touch.append(hit)
gmax["solidity"], gmax["touching"] = sol, touch
gmax["edge"] = touches_edge_flags(gmax, H, W, cfg.sel_edge_px)
gmax["reason"] = np.select(
    [gmax.n_planes < GOLD_MIN_PLANES, gmax.edge, gmax.area_um2 < cfg.sel_min_area_um2,
     gmax.solidity < cfg.sel_min_solidity, gmax.touching],
    ["< min planes", "edge", "below area floor", "cluster: bilobed", "cluster: touching"], "selected")
GOLD = gmax[gmax.reason == "selected"].copy()

GOLD_FLAGS = gmax[gmax.reason.str.startswith("cluster")][
    ["gold_id", "t", "z", "area_um2", "solidity", "touching", "reason", "bbox_min_row", "bbox_min_col",
     "bbox_max_row", "bbox_max_col", "true_time_min"]]
GOLD_FLAGS.to_csv(VALID_DIR / "gold_cluster_flags.csv", index=False)
print("gold maximal sections by outcome:"); display(pd.crosstab(gmax.t, gmax.reason, margins=True))
print(f"cluster-flagged gold ROIs: {len(GOLD_FLAGS)} -> {VALID_DIR / 'gold_cluster_flags.csv'}")
print(f"gold single nuclei: {len(GOLD)} | model single nuclei: {len(single_nuclei_df)}")


## V3. Model vs gold — growth curve equivalence

In [ ]:
# V3a. VALIDATION RESULT: median logistic on model and gold single nuclei, bootstrap (nuclei),
# TOST on A(30)/A(55) with the ±0.5 µm-diameter margin (90 % CI of model − gold inside ±margin),
# and phase velocity differences (no pass/fail until a velocity margin is chosen).
N_BOOT_V = 500
VELOCITY_MARGIN = None
VAL = {}
for name, (t, y) in {"model": (single_nuclei_df.true_time_min, single_nuclei_df.area_um2),
                     "gold": (GOLD.true_time_min, GOLD.area_um2)}.items():
    t, y = np.asarray(t, float), np.asarray(y, float)
    p = fit_median_logistic(t, y)
    VAL[name] = dict(t=t, y=y, p=p, boots=boot_logistic(t, y, p, n=N_BOOT_V, seed=0 if name == "model" else 1))
t_obs = (float(np.floor(VAL["gold"]["t"].min())), float(np.ceil(VAL["gold"]["t"].max())))
VAL_PHASES = phase_windows(VAL["gold"]["p"], t_obs=t_obs, min_len=cfg.frame_interval_min)
for v in VAL.values():
    v["m"] = phase_velocity_table(v["boots"], VAL_PHASES)

display(pd.DataFrame({n: {k: ci_text(s) for k, s in v["m"].items()} for n, v in VAL.items()}))
rows = []
for col in VAL["model"]["m"].columns:
    diff = VAL["model"]["m"][col].to_numpy() - VAL["gold"]["m"][col].to_numpy()
    lo, hi = np.nanquantile(diff, [.05, .95])
    r = {"quantity": col, "model − gold (95% CI)": ci_text(diff), "90% CI": f"[{lo:+.3g}, {hi:+.3g}]"}
    if col in ("A(30)", "A(55)"):
        mg = equivalence_margin_um2(VAL["gold"]["m"][col].median())
        r.update(margin=round(mg, 1), equivalent="yes" if -mg < lo and hi < mg else "no")
    elif col.startswith("abs vel") and VELOCITY_MARGIN is not None:
        r.update(margin=VELOCITY_MARGIN, equivalent="yes" if -VELOCITY_MARGIN < lo and hi < VELOCITY_MARGIN else "no")
    rows.append(r)
EQUIV = pd.DataFrame(rows).set_index("quantity")
display(EQUIV); EQUIV.to_csv(VALID_DIR / "equivalence_model_vs_gold.csv")


In [ ]:
# V3b. Figures: model vs gold curves with bands; absolute and relative velocity.
tt = np.linspace(*t_obs, 300)
fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
for name, c, mk in (("gold", "crimson", "^"), ("model", "teal", "o")):
    v = VAL[name]; cur = logi(v["boots"], tt)
    axs[0].scatter(v["t"], v["y"], s=10, c=c, marker=mk, alpha=.3)
    for ax, g in ((axs[0], cur), (axs[1], np.gradient(cur, tt, axis=1)), (axs[2], np.gradient(np.log(cur), tt, axis=1))):
        ax.fill_between(tt, *np.quantile(g, [.025, .975], axis=0), color=c, alpha=.2)
        ax.plot(tt, np.median(g, axis=0), c=c, lw=2.5, label=f"{name} (n={len(v['t'])})")
for ax, (ttl, yl) in zip(axs, (("area", "µm²"), ("absolute velocity", "dA/dt (µm²/min)"),
                               ("relative velocity", "d lnA/dt (1/min)"))):
    for (ph, win), shade in zip(VAL_PHASES.items(), ("#e8f0ff", "#fff3e0", "#eef7ee")):
        if win: ax.axvspan(*win, color=shade, zorder=0)
    ax.set(title=f"model vs gold — {ttl}", xlabel="Acquisition time (minutes)", ylabel=yl); ax.grid(alpha=.2)
axs[0].legend(); plt.tight_layout(); fig.savefig(VALID_DIR / "model_vs_gold.png", dpi=300); plt.show()


## V4. Matched decomposition of the population gap

In [ ]:
# V4. DIAGNOSTIC (informational): where does a population gap come from? Match model and gold
# single nuclei (same t, IoU of maximal-section footprints, Hungarian per t) and split the
# A(30)/A(55) gap into (a) model nuclei without a gold match, (b) boundary bias on matched pairs,
# (c) gold nuclei without a model match. The three rows add up to the total gap.
from scipy.optimize import linear_sum_assignment
MATCH_IOU = 0.30
M = single_nuclei_df.copy()
M["box"] = list(zip(M.bbox_min_row, M.bbox_min_col, M.bbox_max_row, M.bbox_max_col))
M["mask"] = [np.asarray(inst[int(r.t), int(r.z), b[0]:b[2], b[1]:b[3]]) == int(r.label)
             for r, b in zip(M.itertuples(), M.box)]
Gm = GOLD.copy()
Gm["mask"] = [rois[i]["mask"] for i in Gm.roi_index]


def _iou(bA, mA, bB, mB):
    y0, x0 = max(bA[0], bB[0]), max(bA[1], bB[1]); y1, x1 = min(bA[2], bB[2]), min(bA[3], bB[3])
    if y0 >= y1 or x0 >= x1:
        return 0.0
    inter = (mA[y0-bA[0]:y1-bA[0], x0-bA[1]:x1-bA[1]] & mB[y0-bB[0]:y1-bB[0], x0-bB[1]:x1-bB[1]]).sum()
    return inter / (mA.sum() + mB.sum() - inter)


pairs = []
for t in sorted(set(M.t) & set(Gm.t)):
    mi, gi = M.index[M.t == t], Gm.index[Gm.t == t]
    Q = np.zeros((len(mi), len(gi)))
    for a, i in enumerate(mi):
        for b_, j in enumerate(gi):
            bm, bg = M.at[i, "box"], Gm.at[j, "box"]
            if bm[0] < bg[2] and bg[0] < bm[2] and bm[1] < bg[3] and bg[1] < bm[3]:
                Q[a, b_] = _iou(bm, M.at[i, "mask"], bg, Gm.at[j, "mask"])
    rr, cc = linear_sum_assignment(-Q)
    pairs += [(mi[a], gi[b_], Q[a, b_]) for a, b_ in zip(rr, cc) if Q[a, b_] >= MATCH_IOU]
P = pd.DataFrame(pairs, columns=["m", "g", "iou"])
P["model_um2"] = M.loc[P.m, "area_um2"].to_numpy(); P["gold_um2"] = Gm.loc[P.g, "area_um2"].to_numpy()
P["time"] = Gm.loc[P.g, "true_time_min"].to_numpy(); P["ratio"] = P.model_um2 / P.gold_um2
P["same_z"] = M.loc[P.m, "z"].to_numpy() == Gm.loc[P.g, "z"].to_numpy()
M["matched"] = M.index.isin(P.m); Gm["matched"] = Gm.index.isin(P.g)
print(f"matched {len(P)} | model unmatched {(~M.matched).sum()} / {len(M)} | gold unmatched "
      f"{(~Gm.matched).sum()} / {len(Gm)} | same plane in {P.same_z.mean():.0%} of pairs | median ratio {P.ratio.median():.3f}")


def _A(xx, yy):
    p = fit_median_logistic(np.asarray(xx, float), np.asarray(yy, float))
    return float(logi(p, 30.)), float(logi(p, 55.))
F = pd.DataFrame({"model, all": _A(M.true_time_min, M.area_um2),
                  "model, matched": _A(M.true_time_min[M.matched], M.area_um2[M.matched]),
                  "gold, matched": _A(Gm.true_time_min[Gm.matched], Gm.area_um2[Gm.matched]),
                  "gold, all": _A(Gm.true_time_min, Gm.area_um2)}, index=["A(30)", "A(55)"]).T
DECOMP = pd.DataFrame({
    "(a) model extras": F.loc["model, all"] - F.loc["model, matched"],
    "(b) boundary bias (matched pairs)": F.loc["model, matched"] - F.loc["gold, matched"],
    "(c) missed gold": F.loc["gold, matched"] - F.loc["gold, all"],
    "total gap": F.loc["model, all"] - F.loc["gold, all"]}).T.round(1)
display(F.round(1)); display(DECOMP); DECOMP.to_csv(VALID_DIR / "gap_decomposition.csv")

fig, axs = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
axs[0].scatter(Gm.true_time_min, Gm.area_um2, s=10, c="0.8", label="gold, matched")
axs[0].scatter(Gm.true_time_min[~Gm.matched], Gm.area_um2[~Gm.matched], s=22, c="crimson",
               label=f"gold, no model match (n={(~Gm.matched).sum()})")
axs[1].scatter(M.true_time_min, M.area_um2, s=10, c="0.8", label="model, matched")
axs[1].scatter(M.true_time_min[~M.matched], M.area_um2[~M.matched], s=22, c="teal",
               label=f"model, no gold match (n={(~M.matched).sum()})")
axs[2].scatter(P.gold_um2, P.model_um2, s=12, c=P.time, cmap="viridis")
lim = [0, max(P.gold_um2.max(), P.model_um2.max()) * 1.05]
axs[2].plot(lim, lim, "k--", lw=1); axs[2].set(xlim=lim, xlabel="gold area (µm²)", title="matched pairs (colour = time)")
for ax in axs[:2]:
    ax.set(xlabel="Acquisition time (minutes)"); ax.legend(); ax.grid(alpha=.2)
axs[0].set_ylabel("area (µm²)"); plt.tight_layout(); plt.show()


## V5. Detection audits

In [ ]:
# V5a. DIAGNOSTIC (informational): how does the model do AT EACH GOLD EQUATOR? On the plane of each
# gold single nucleus's maximal section: fraction of the gold outline covered by model instances and
# how many instances share it. Classes: OK (>= 85 % by one label) / split (>= 85 % by 2+ labels) /
# under-seg (30–85 %) / missed (< 30 %). Uses ALL saved instances, not only selected nuclei.
rows = []
for _, g in GOLD.iterrows():
    (y0, x0, y1, x1), gm = rois[g.roi_index]["box"], rois[g.roi_index]["mask"]
    lab = np.asarray(inst[int(g.t), int(g.z), y0:y1, x0:x1])[gm]
    ids, cnt = np.unique(lab[lab > 0], return_counts=True)
    cov = cnt.sum() / gm.sum(); one = (cnt.max() / gm.sum()) if len(cnt) else 0.0
    big = int((cnt >= 0.10 * gm.sum()).sum())
    cls = ("missed" if cov < .30 else "under-seg" if cov < .85 else "split" if (big >= 2 and one < .85) else "OK")
    rows.append(dict(gold_id=g.gold_id, roi_index=g.roi_index, t=g.t, z=g.z, time=g.true_time_min,
                     gold_um2=g.area_um2, cov=cov, one_label=one, n_labels=big, cls=cls))
EQ = pd.DataFrame(rows)
EQ["late"] = EQ.time >= 30
display(EQ.groupby(["late", "cls"]).agg(n=("cls", "size"), median_cover=("cov", "median")).round(2))
EQ.to_csv(VALID_DIR / "equator_classes.csv", index=False)


In [ ]:
# V5b. DIAGNOSTIC (informational): model single nuclei without a gold match (V4). For each, how much
# of it is covered by any gold outline ON ITS OWN PLANE, and where its z sits relative to the gold
# column it lies in. "NO gold on this plane" = the model calls nucleus where none was traced.
def _cov(bm, mk, bg, mg):
    y0, x0, y1, x1 = max(bm[0], bg[0]), max(bm[1], bg[1]), min(bm[2], bg[2]), min(bm[3], bg[3])
    if y0 >= y1 or x0 >= x1:
        return 0
    return (mk[y0-bm[0]:y1-bm[0], x0-bm[1]:x1-bm[1]] & mg[y0-bg[0]:y1-bg[0], x0-bg[1]:x1-bg[1]]).sum()

zr = G.groupby("gold_id").z.agg(["min", "max"])
X = M[~M.matched].copy()
same, zmin, zmax = [], [], []
for _, r in X.iterrows():
    hits = [(k, _cov(r["box"], r["mask"], rois[k]["box"], rois[k]["mask"])) for k in gz.get((r["t"], r["z"]), [])]
    k, inter = max(hits, key=lambda h: h[1], default=(None, 0))
    same.append(inter / r["mask"].sum())
    gid = G.at[k, "gold_id"] if (k is not None and inter > 0) else None
    zmin.append(zr.at[gid, "min"] if gid is not None else np.nan); zmax.append(zr.at[gid, "max"] if gid is not None else np.nan)
X["same_plane_gold_cov"], X["gold_zmin"], X["gold_zmax"] = same, zmin, zmax
X["verdict"] = np.where(X.same_plane_gold_cov >= .5, "gold traced this plane",
               np.where(X.same_plane_gold_cov > .1, "partial gold on this plane", "NO gold on this plane"))
display(X.verdict.value_counts().to_frame())
X.drop(columns=["mask"]).to_csv(VALID_DIR / "model_unmatched.csv", index=False)

PAD = 40
show = pd.concat([X[X.verdict == "NO gold on this plane"], X[X.verdict != "NO gold on this plane"]]).head(24)
fig, axs = plt.subplots(4, 6, figsize=(18, 12)); axs = axs.ravel()
for ax in axs: ax.axis("off")
for ax, (_, r) in zip(axs, show.iterrows()):
    y0, x0, y1, x1 = r["box"]; Y0, X0, Y1, X1 = max(0, y0 - PAD), max(0, x0 - PAD), min(H, y1 + PAD), min(W, x1 + PAD)
    im = np.asarray(img_5d[int(r["t"]), int(r["z"]), cfg.nuclear_channel_index, Y0:Y1, X0:X1], np.float32)
    ax.imshow(im, cmap="gray", vmin=np.percentile(im, 1), vmax=np.percentile(im, 99.5))
    full = np.zeros(im.shape, bool); full[y0-Y0:y1-Y0, x0-X0:x1-X0] = r["mask"]
    for c in measure.find_contours(full.astype(float), .5): ax.plot(c[:, 1], c[:, 0], c="cyan", lw=1)
    for k in gz.get((r["t"], r["z"]), []):
        gy0, gx0, gy1, gx1 = rois[k]["box"]
        if gy1 <= Y0 or gy0 >= Y1 or gx1 <= X0 or gx0 >= X1:
            continue
        canvas = np.zeros((max(Y1, gy1) - min(Y0, gy0), max(X1, gx1) - min(X0, gx0)), bool)
        oy, ox = min(Y0, gy0), min(X0, gx0)
        canvas[gy0 - oy:gy1 - oy, gx0 - ox:gx1 - ox] = rois[k]["mask"]
        for c in measure.find_contours(canvas[Y0 - oy:Y1 - oy, X0 - ox:X1 - ox].astype(float), .5):
            ax.plot(c[:, 1], c[:, 0], c="red", lw=1)
    ax.set_title(f"t{int(r['t'])} z{int(r['z'])} {r['area_um2']:.0f} µm²\n{r['verdict']}", fontsize=7)
plt.suptitle("Unmatched model nuclei: NLS at their plane, model (cyan), gold on that plane (red)")
plt.tight_layout(); plt.show()


## V6. Growth-model family comparison

In [ ]:
# V6. DIAGNOSTIC (informational): growth-model family comparison on model and gold single nuclei.
# Fit in log space (robust loss); judge by leave-one-frame-out CV (median |log error| on the held-out
# frame) and AIC. The pipeline uses the median logistic (chosen this way on 2026-11-04).
MODEL_FAMILIES = {
    "saturating exp": (lambda p, t: p[0] - (p[0] - p[1]) * np.exp(-p[2] * (t - 6)), [450, 80, .03], [50, 1, 1e-4], [3000, 500, 1]),
    "logistic":       (lambda p, t: p[0] / (1 + np.exp(-p[1] * (t - p[2]))), [420, .08, 20], [50, 1e-3, -60], [3000, 1, 120]),
    "Gompertz":       (lambda p, t: p[0] * np.exp(-np.exp(-p[1] * (t - p[2]))), [430, .06, 15], [50, 1e-3, -60], [3000, 1, 120]),
    "power law":      (lambda p, t: p[0] * np.clip(t - p[2], 1e-3, None) ** p[1], [40, .6, 0], [1e-3, .05, -60], [1e4, 3, 5.9]),
    "linear":         (lambda p, t: p[0] + p[1] * t, [50, 6], [-500, 0], [1000, 50]),
}

def _fit_log(f, p0, lo, hi, t, y):
    return least_squares(lambda p: np.log(np.clip(f(p, t), 1e-6, None)) - np.log(y), p0,
                         bounds=(lo, hi), loss="soft_l1", f_scale=0.15).x

out = []
for label, v in VAL.items():
    t, y = v["t"], v["y"]; frame = np.floor(t / cfg.frame_interval_min).astype(int)
    for name, (f, p0, lo, hi) in MODEL_FAMILIES.items():
        p = _fit_log(f, p0, lo, hi, t, y); res = np.log(f(p, t)) - np.log(y)
        cv = [np.median(np.abs(np.log(f(_fit_log(f, p, lo, hi, t[frame != fr], y[frame != fr]), t[frame == fr]))
                               - np.log(y[frame == fr]))) for fr in np.unique(frame)]
        out.append(dict(population=label, model=name, k=len(p0),
                        AIC=len(y) * np.log(np.mean(res ** 2)) + 2 * len(p0), CV=np.mean(cv),
                        **{"A(30)": f(p, 30.), "A(55)": f(p, 55.)}))
FAM = pd.DataFrame(out)
FAM["ΔAIC"] = FAM.AIC - FAM.groupby("population").AIC.transform("min")
display(FAM.set_index(["population", "model"]).round(3).sort_index())
FAM.to_csv(VALID_DIR / "growth_model_families.csv", index=False)


## V7. Selection-rule sensitivity

In [ ]:
# V7. DIAGNOSTIC (informational, saves nothing): sensitivity of the validation result to each
# selection rule. Re-runs the pipeline's select_single_nuclei on this run's objects with ONE setting
# changed at a time (copy of cfg; the pipeline's tables are untouched) and refits A(30)/A(55).
# Gold rules follow the same settings so both sides stay symmetric only for the edge/area rules;
# the cluster rule on gold is fixed (V2). Expect a few minutes per row.
import copy
objects_df = pd.read_pickle(cfg.obj_dir / "plane_objects.pkl")
VARIANTS = [("as configured", {}), ("window ±1", {"sel_window_planes": 1}), ("window ±3", {"sel_window_planes": 3}),
            ("solidity 0.85", {"sel_min_solidity": 0.85}), ("solidity 0.93", {"sel_min_solidity": 0.93}),
            ("touch 2 px", {"sel_touch_px": 2}), ("touch 5 px", {"sel_touch_px": 5}),
            ("area floor 30", {"sel_min_area_um2": 30.0}), ("area floor 80", {"sel_min_area_um2": 80.0})]
g30, g55 = _A(GOLD.true_time_min, GOLD.area_um2)
rows = []
for name, change in VARIANTS:
    c = copy.copy(cfg)
    for k, v in change.items(): setattr(c, k, v)
    _, nuc = select_single_nuclei(objects_df, POPULATION["components"], H, W, c)
    nuc = add_tile_timing_metadata(nuc, (H, W), c)
    a30, a55 = _A(nuc.true_time_min, nuc.area_um2)
    rows.append(dict(variant=name, n=len(nuc), **{"A(30)": a30, "A(55)": a55,
                     "A(30) − gold": a30 - g30, "A(55) − gold": a55 - g55}))
    print(rows[-1])
SENS = pd.DataFrame(rows).set_index("variant").round(1)
display(SENS); SENS.to_csv(VALID_DIR / "selection_sensitivity.csv")
print(f"gold: A(30) {g30:.1f}, A(55) {g55:.1f} | margins ±{equivalence_margin_um2(g30):.1f}, ±{equivalence_margin_um2(g55):.1f}")


## V8. Vulcan 3.1.2 hard examples

In [ ]:
# V8. Vulcan 3.1.2 HARD-EXAMPLE EXPORT (writes one CSV; changes nothing else).
# Three kinds, each with its region in frame coordinates:
#   under_segmented_equator : gold equators the model covers 30–85 % (V5a) -> positive examples
#   missed_equator          : gold equators the model covers < 30 % (V5a)   -> positive examples
#   false_positive          : model single nuclei with NO gold outline on their plane (V5b) -> negatives
# Stitching ghosts (a duplicated nucleus in a tile-overlap region, e.g. t6 z13) are not detected
# automatically: add them by hand to MANUAL_HARD_EXAMPLES as (kind, t, z, y0, x0, y1, x1, note).
MANUAL_HARD_EXAMPLES = [
    # ("stitching_ghost", 6, 13, y0, x0, y1, x1, "duplicated nucleus at tile seam"),
]
rows = []
for kind, sel in (("under_segmented_equator", EQ.cls == "under-seg"), ("missed_equator", EQ.cls == "missed")):
    for _, e in EQ[sel].iterrows():
        y0, x0, y1, x1 = rois[e.roi_index]["box"]
        rows.append(dict(kind=kind, t=int(e.t), z=int(e.z), y0=y0, x0=x0, y1=y1, x1=x1,
                         area_um2=e.gold_um2, note=f"gold coverage {e['cov']:.0%}"))
for _, r in X[X.verdict == "NO gold on this plane"].iterrows():
    y0, x0, y1, x1 = r["box"]
    rows.append(dict(kind="false_positive", t=int(r.t), z=int(r.z), y0=y0, x0=x0, y1=y1, x1=x1,
                     area_um2=r.area_um2, note="no gold outline on this plane"))
rows += [dict(zip(["kind", "t", "z", "y0", "x0", "y1", "x1", "note"], m)) for m in MANUAL_HARD_EXAMPLES]
HARD = pd.DataFrame(rows)
HARD["source_run"] = run.run_id; HARD["model_sha256"] = cfg.model_sha256
HARD.to_csv(VALID_DIR / "vulcan_3_1_2_hard_examples.csv", index=False)
display(HARD.kind.value_counts().to_frame())
print("->", VALID_DIR / "vulcan_3_1_2_hard_examples.csv")
